# Intelligent Customer Support Chatbot
**CA2 Project | Natural Language Processing**

Dataset: `customer_support_dataset.csv` — 40 queries, 8 intent classes
---

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, string, warnings, pickle
warnings.filterwarnings('ignore')

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report)
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries loaded.')

PAL = {
    'order_tracking':'#2E86AB','returns_refunds':'#A23B72',
    'order_cancellation':'#F18F01','account_support':'#C73E1D',
    'payment_issues':'#44BBA4','promotions':'#E94F37',
    'general_inquiry':'#393E41','product_complaint':'#8B5E83'
}

---
## 2. Load and Explore the Dataset

In [ ]:
df = pd.read_csv('customer_support_dataset.csv')
print('Shape:', df.shape)
print('Columns:', list(df.columns))
print('Missing values:', df.isnull().sum().sum())
print('Duplicates:', df.duplicated().sum())
df.head(10)

In [ ]:
print('Intent Distribution:')
print(df['intent'].value_counts())
print(f'\nTotal unique intents: {df["intent"].nunique()}')
print(f'Samples per intent: {df.shape[0]//df["intent"].nunique()} (perfectly balanced)')

In [ ]:
# Quick look at one sample per intent
for intent in sorted(df['intent'].unique()):
    row = df[df['intent']==intent].iloc[0]
    print(f'[{intent}]')
    print(f'  Q: {row["query"]}')
    print(f'  A: {row["response"][:90]}...')
    print()

---
## 3. Exploratory Data Analysis

In [ ]:
df['word_count'] = df['query'].apply(lambda x: len(x.split()))
df['char_count'] = df['query'].apply(len)
df['resp_words'] = df['response'].apply(lambda x: len(x.split()))
print('Basic stats:')
print(df[['word_count','char_count','resp_words']].describe().round(2))

In [ ]:
# Fig 1 – Intent distribution
ic = df['intent'].value_counts().reindex(sorted(df['intent'].unique()))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Intent Distribution', fontsize=14, fontweight='bold')

bars = ax1.bar(range(len(ic)), ic.values, color=[PAL[i] for i in ic.index],
               width=0.6, edgecolor='white', linewidth=1.5)
ax1.set_xticks(range(len(ic)))
ax1.set_xticklabels([i.replace('_','\n') for i in ic.index], fontsize=9)
ax1.set_ylabel('Number of Queries'); ax1.set_ylim(0, 7.5)
for bar in bars:
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1,
             str(int(bar.get_height())), ha='center', fontweight='bold', fontsize=11)

ax2.pie(ic.values, labels=[i.replace('_','\n') for i in ic.index],
        autopct='%1.0f%%', colors=[PAL[i] for i in ic.index],
        startangle=90, wedgeprops=dict(edgecolor='white',linewidth=2))
plt.tight_layout(); plt.show()

In [ ]:
# Fig 2 – Length analysis
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Query Length Analysis', fontsize=13, fontweight='bold')

axes[0].hist(df['word_count'], bins=range(4,16), color='#2E86AB', edgecolor='white', rwidth=0.85)
axes[0].axvline(df['word_count'].mean(), color='red', ls='--', lw=2,
                label=f'Mean={df["word_count"].mean():.1f}')
axes[0].set_title('Word Count'); axes[0].legend()

axes[1].hist(df['char_count'], bins=10, color='#A23B72', edgecolor='white')
axes[1].axvline(df['char_count'].mean(), color='orange', ls='--', lw=2,
                label=f'Mean={df["char_count"].mean():.1f}')
axes[1].set_title('Char Count'); axes[1].legend()

avg_wc = df.groupby('intent')['word_count'].mean().sort_values()
axes[2].barh(range(len(avg_wc)), avg_wc.values, color=[PAL[i] for i in avg_wc.index])
axes[2].set_yticks(range(len(avg_wc)))
axes[2].set_yticklabels([i.replace('_',' ') for i in avg_wc.index], fontsize=9)
axes[2].set_title('Avg Words per Intent')
for i,v in enumerate(avg_wc.values): axes[2].text(v+0.05,i,f'{v:.1f}',va='center')

plt.tight_layout(); plt.show()

---
## 4. Text Preprocessing

In [ ]:
def preprocess(text):
    """Lowercase, remove punctuation and digits, strip extra spaces."""
    text = text.lower()
    text = text.translate(str.maketrans('','',string.punctuation))
    text = re.sub(r'\d+','',text)
    return re.sub(r'\s+',' ',text).strip()

df['clean_query'] = df['query'].apply(preprocess)

print('Before and After preprocessing:')
for _, row in df[['query','clean_query']].head(6).iterrows():
    print(f'  IN : {row["query"]}')
    print(f'  OUT: {row["clean_query"]}')
    print()

---
## 5. Feature Extraction — TF-IDF

In [ ]:
le = LabelEncoder()
df['label'] = le.fit_transform(df['intent'])
print('Label mapping:')
for i,c in enumerate(le.classes_): print(f'  {i} -> {c}')

tfidf = TfidfVectorizer(max_features=500, ngram_range=(1,2),
                         stop_words='english', sublinear_tf=True)
X = tfidf.fit_transform(df['clean_query'])
y = df['label']
print(f'\nFeature matrix: {X.shape}')

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train samples: {X_tr.shape[0]}, Test samples: {X_te.shape[0]}')

In [ ]:
# Fig 3 – TF-IDF terms per intent
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle('Top TF-IDF Terms per Intent', fontsize=14, fontweight='bold')
for idx, intent in enumerate(sorted(df['intent'].unique())):
    subset = df[df['intent']==intent]['clean_query']
    tv = TfidfVectorizer(max_features=20, ngram_range=(1,2), stop_words='english')
    tv.fit(subset)
    arr = tv.transform(subset).toarray()
    ms = arr.mean(axis=0)
    si = np.argsort(ms)[::-1][:8]
    terms = [tv.get_feature_names_out()[i] for i in si]
    vals  = [ms[i] for i in si]
    ax = axes.flatten()[idx]
    ax.barh(range(len(terms)), vals[::-1], color=PAL[intent], edgecolor='white')
    ax.set_yticks(range(len(terms)))
    ax.set_yticklabels(terms[::-1], fontsize=9)
    ax.set_title(intent.replace('_',' ').title(), fontweight='bold', fontsize=10)
    ax.set_xlabel('Mean TF-IDF')
plt.tight_layout(); plt.show()

---
## 6. Model Training and Evaluation

In [ ]:
MODELS = {
    'Logistic Regression' : LogisticRegression(max_iter=500, C=1.0, random_state=42),
    'SVM (Linear)'        : SVC(kernel='linear', probability=True, random_state=42),
    'Random Forest'       : RandomForestClassifier(n_estimators=150, random_state=42),
    'Naive Bayes'         : MultinomialNB(alpha=0.5),
    'Gradient Boosting'   : GradientBoostingClassifier(n_estimators=100, random_state=42),
}

RES = {}
print(f'{"Model":<24} {"Acc":>6} {"F1":>6} {"Prec":>6} {"Rec":>6}')
print('-'*55)
for name, model in MODELS.items():
    model.fit(X_tr, y_tr)
    p = model.predict(X_te)
    RES[name] = {
        'acc' : accuracy_score(y_te, p),
        'f1'  : f1_score(y_te, p, average='weighted', zero_division=0),
        'prec': precision_score(y_te, p, average='weighted', zero_division=0),
        'rec' : recall_score(y_te, p, average='weighted', zero_division=0),
        'pred': p, 'model': model
    }
    r = RES[name]
    print(f'{name:<24} {r["acc"]:>6.3f} {r["f1"]:>6.3f} {r["prec"]:>6.3f} {r["rec"]:>6.3f}')

best_name = max(RES, key=lambda x: RES[x]['f1'])
best_pred = RES[best_name]['pred']
best_model = RES[best_name]['model']
print(f'\nBest model: {best_name}')

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
CV_RES = {}
print('5-Fold Cross-Validation:')
print('-'*55)
for name, model in MODELS.items():
    pipe = Pipeline([
        ('tfidf', TfidfVectorizer(max_features=500, ngram_range=(1,2),
                                   stop_words='english', sublinear_tf=True)),
        ('clf',   model)
    ])
    sc = cross_val_score(pipe, df['clean_query'], df['intent'], cv=skf, scoring='accuracy')
    CV_RES[name] = sc
    print(f'{name:<24}: Mean={sc.mean():.4f}  Std={sc.std():.4f}  Folds={np.round(sc,3)}')

In [ ]:
# Fig 4 – Model comparison
model_names = list(RES.keys())
MC = ['#2E86AB','#A23B72','#F18F01','#44BBA4']
BC = ['#2E86AB','#A23B72','#F18F01','#C73E1D','#44BBA4']

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold')

x = np.arange(len(model_names)); w = 0.2
for i,(key,lbl,c) in enumerate(zip(['acc','f1','prec','rec'],
    ['Accuracy','F1','Precision','Recall'],MC)):
    vals = [RES[n][key] for n in model_names]
    axes[0].bar(x+i*w, vals, w, label=lbl, color=c, edgecolor='white')
axes[0].set_xticks(x+1.5*w)
axes[0].set_xticklabels([n.replace(' ','\n') for n in model_names], fontsize=9)
axes[0].set_ylim(0,1.2); axes[0].legend()
axes[0].set_title('Test Set Metrics')

cv_means = [CV_RES[n].mean() for n in model_names]
cv_stds  = [CV_RES[n].std()  for n in model_names]
bars = axes[1].bar(x, cv_means, 0.55, yerr=cv_stds, capsize=7,
                   color=BC, edgecolor='white', error_kw={'elinewidth':2})
axes[1].set_xticks(x)
axes[1].set_xticklabels([n.replace(' ','\n') for n in model_names], fontsize=9)
axes[1].set_ylim(0,1.15)
axes[1].set_title('5-Fold CV Accuracy')
for bar,m,s in zip(bars,cv_means,cv_stds):
    axes[1].text(bar.get_x()+bar.get_width()/2, m+s+0.02, f'{m:.3f}',
                 ha='center', fontsize=10, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
print(f'Classification Report — {best_name}')
print(classification_report(y_te, best_pred, target_names=le.classes_, zero_division=0))

In [ ]:
# Fig 5 – Confusion Matrix
cm = confusion_matrix(y_te, best_pred)
fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=[l.replace('_','\n') for l in le.classes_],
            yticklabels=[l.replace('_','\n') for l in le.classes_],
            linewidths=0.6, annot_kws={'size':13,'weight':'bold'})
ax.set_title(f'Confusion Matrix — {best_name}', fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
plt.tight_layout(); plt.show()

---
## 7. Building the Chatbot

In [ ]:
class CustomerSupportChatbot:
    """
    Retrieval-based customer support chatbot.
    Intent classification via TF-IDF + ML, response retrieval via cosine similarity.
    """
    GREETS   = ['hi','hello','hey','good morning','good evening','howdy','hii']
    FAREWELLS = ['bye','goodbye','see you','take care','exit','quit']
    THANKS    = ['thank you','thanks','thank u','ty','thx','cheers']

    GREET_RESP = (
        'Hello! Welcome to Customer Support.\n'
        'I can help with: order tracking, returns & refunds, order cancellation,\n'
        'payment issues, account support, promotions, or product complaints.\n'
        'What can I help you with today?'
    )
    BYE_RESP   = 'Thank you for contacting us. Have a great day!'
    TYVM_RESP  = "You're welcome! Is there anything else I can help you with?"
    FALLBACK   = (
        "I'm sorry, I couldn't quite understand that.\n"
        'Please try rephrasing, or choose a topic: order tracking | return item | '
        'cancel order | payment | account | discounts | complaint'
    )
    THRESHOLD = 0.30

    def __init__(self, df, tfidf, model, le):
        self.df   = df.copy()
        self.tfidf = tfidf
        self.model = model
        self.le    = le
        self.history = []
        self._corpus_vecs = tfidf.transform(df['clean_query'])

    def _clean(self, text):
        text = text.lower().translate(str.maketrans('','',string.punctuation))
        return re.sub(r'\s+',' ', re.sub(r'\d+','',text)).strip()

    def _special(self, raw):
        t = raw.lower().strip()
        if any(g in t for g in self.GREETS):    return 'greet'
        if any(f in t for f in self.FAREWELLS):  return 'bye'
        if any(k in t for k in self.THANKS):     return 'thanks'
        return None

    def _classify(self, cleaned):
        v = self.tfidf.transform([cleaned])
        proba = self.model.predict_proba(v)[0]
        idx = np.argmax(proba)
        return self.le.inverse_transform([idx])[0], proba[idx]

    def _retrieve(self, cleaned, intent):
        sub = self.df[self.df['intent']==intent].reset_index(drop=True)
        if sub.empty: return self.FALLBACK
        vecs = self.tfidf.transform(sub['clean_query'])
        sims = cosine_similarity(self.tfidf.transform([cleaned]), vecs).flatten()
        return sub.iloc[np.argmax(sims)]['response']

    def respond(self, user_input):
        sp = self._special(user_input)
        if   sp=='greet' : resp,intent,conf = self.GREET_RESP,'greeting',1.0
        elif sp=='bye'   : resp,intent,conf = self.BYE_RESP,'farewell',1.0
        elif sp=='thanks': resp,intent,conf = self.TYVM_RESP,'thanks',1.0
        else:
            clean = self._clean(user_input)
            intent, conf = self._classify(clean)
            resp = self._retrieve(clean, intent) if conf >= self.THRESHOLD else self.FALLBACK
        self.history.append({'user': user_input, 'intent': intent,
                              'confidence': conf, 'bot': resp})
        return resp, intent, conf

    def reset(self): self.history = []

bot = CustomerSupportChatbot(df, tfidf, best_model, le)
print('Chatbot ready. Model:', best_name)

---
## 8. Chatbot Demo

In [ ]:
test_queries = [
    'Hello!',
    'Where is my order?',
    'I want to return a damaged product',
    'My payment was declined at checkout',
    'Can you please cancel my order?',
    'I forgot my login password',
    'Do you have any discount coupons right now?',
    'I got a completely wrong item in my delivery',
    'How many days does shipping take?',
    'asdfgh random gibberish',
    'Thank you so much!',
    'Bye!'
]

bot.reset()
print('='*70)
print('            CUSTOMER SUPPORT CHATBOT — DEMO')
print('='*70)
for q in test_queries:
    resp, intent, conf = bot.respond(q)
    bar = '|' * int(conf * 10) + '.' * (10 - int(conf*10))
    print(f'\nUser   : {q}')
    print(f'Intent : {intent}  [{bar}]  {conf:.0%}')
    print(f'Bot    : {resp[:120]}')
    print('-'*70)

In [ ]:
def interactive_chat():
    """Run a live interactive chat session in Jupyter."""
    bot.reset()
    print('='*60)
    print('  Customer Support Chatbot  |  type quit to exit')
    print('='*60)
    while True:
        user_input = input('\nYou: ').strip()
        if not user_input: continue
        resp, intent, conf = bot.respond(user_input)
        print(f'Bot [{intent} | {conf:.0%}]: {resp}')
        if user_input.lower() in ['quit','exit','bye','goodbye']: break

# Uncomment to run:
# interactive_chat()

---
## 9. Transformer Concepts — Self-Attention Visualization

In [ ]:
# Simulate self-attention for the query 'Where is my order package?'
tokens = ['where','is','my','order','package']
np.random.seed(7)
Q = np.random.randn(5,8); K = np.random.randn(5,8)
raw = (Q @ K.T) / np.sqrt(8)
def softmax2d(M):
    e = np.exp(M - M.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)
attn = softmax2d(raw)

fig, ax = plt.subplots(figsize=(7,6))
sns.heatmap(attn, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            xticklabels=tokens, yticklabels=tokens,
            linewidths=0.5, linecolor='white', annot_kws={'size':12,'weight':'bold'})
ax.set_title('Self-Attention Matrix — "Where is my order package?"',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Key Token (attended to)')
ax.set_ylabel('Query Token (attending from)')
plt.tight_layout(); plt.show()
print('Darker cells = stronger attention between that token pair.')

---
## 10. Save the Model

In [ ]:
with open('chatbot_model.pkl','wb') as f:
    pickle.dump({'tfidf':tfidf,'model':best_model,'le':le,'model_name':best_name}, f)
print('Model saved to chatbot_model.pkl')
print(f'Best model   : {best_name}')
print(f'Test accuracy: {RES[best_name]["acc"]:.4f}')
print(f'Weighted F1  : {RES[best_name]["f1"]:.4f}')
print(f'CV mean      : {CV_RES[best_name].mean():.4f}')